# Démonstration du Fine-tuning LoRA DataCenter

Ce notebook démontre le pipeline complet de fine-tuning avec LoRA pour adapter Qwen2.5-0.5B aux tâches DataCenter.

**Objectifs:**
- Charger le modèle de base et le dataset
- Configurer les paramètres LoRA
- Entraîner le modèle sur les données DataCenter
- Évaluer les améliorations avant/après fine-tuning
- Sauvegarder les adapters LoRA

## 1. Setup et imports

In [ ]:
import os
import json
import torch
import transformers
from pathlib import Path
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Chemins
BASE_DIR = Path('/vercel/share/v0-project')
DATA_DIR = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'models'
ADAPTER_DIR = MODEL_DIR / 'adapters'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nPaths:")
print(f"  Data: {DATA_DIR}")
print(f"  Adapters: {ADAPTER_DIR}")

## 2. Configuration LoRA

In [ ]:
# Configuration LoRA optimisée pour Qwen2.5-0.5B
lora_config = LoraConfig(
    r=16,  # LoRA rank - équilibre entre efficacité et qualité
    lora_alpha=32,  # LoRA scaling factor
    target_modules=['q_proj', 'v_proj'],  # Adapter les projections clé/valeur
    lora_dropout=0.05,  # Dropout pour régularisation
    bias='none',
    task_type='CAUSAL_LM',
    modules_to_save=['lm_head']  # Garder lm_head adaptable
)

print("Configuration LoRA:")
print(f"  r (rank): {lora_config.r}")
print(f"  lora_alpha: {lora_config.lora_alpha}")
print(f"  target_modules: {lora_config.target_modules}")
print(f"  lora_dropout: {lora_config.lora_dropout}")

## 3. Charger le modèle de base

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Chargement du modèle: {MODEL_NAME}")

# Charger le tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer chargé (vocab size: {len(tokenizer)})")

# Charger le modèle avec quantization 4-bit (économise mémoire)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"Chargement du modèle (4-bit quantization)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print(f"✓ Modèle chargé")
print(f"  Paramètres: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# Préparer le modèle pour LoRA
if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, lora_config)
print(f"✓ LoRA appliqué au modèle")
print(f"  Paramètres entraînables: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f}M")

## 4. Charger et préparer le dataset

In [ ]:
# Charger le dataset
train_file = DATA_DIR / 'train.jsonl'

if not train_file.exists():
    print(f"⚠ Fichier dataset non trouvé: {train_file}")
    print(f"  Assurez-vous d'avoir exécuté 02_tests_dataset.ipynb")
else:
    # Compter les exemples
    with open(train_file, 'r') as f:
        num_examples = sum(1 for _ in f)
    
    print(f"✓ Dataset chargé: {train_file}")
    print(f"  Nombre d'exemples: {num_examples}")
    
    # Afficher un exemple
    with open(train_file, 'r') as f:
        first_example = json.loads(f.readline())
        print(f"\nExemple (premiers 200 caractères):")
        print(first_example['text'][:200] + "...")

## 5. Configuration d'entraînement

In [ ]:
# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR / 'checkpoint'),
    overwrite_output_dir=True,
    num_train_epochs=3,  # 3 epochs d'entraînement
    per_device_train_batch_size=2 if torch.cuda.is_available() else 1,  # Adapter selon GPU
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    max_grad_norm=0.3,
    weight_decay=0.001,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=42,
    fp16=torch.cuda.is_available(),
)

print("Configuration d'entraînement:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")

## 6. Lancer le fine-tuning

In [ ]:
# Créer le trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_file,  # Chemin vers le fichier JSONL
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",  # Champ à utiliser dans JSONL
    max_seq_length=512,
    packing=False,  # Chaque exemple est complet
)

print("Trainer créé. Démarrage du fine-tuning...\n")

# Entraîner
train_result = trainer.train()

print(f"\n✓ Fine-tuning terminé!")
print(f"  Perte finale: {train_result.training_loss:.4f}")

## 7. Sauvegarder l'adapter LoRA

In [ ]:
# Sauvegarder l'adapter LoRA
adapter_save_path = ADAPTER_DIR / 'qwen-datacenter-lora'
model.save_pretrained(str(adapter_save_path))

# Sauvegarder également le tokenizer
tokenizer.save_pretrained(str(adapter_save_path))

print(f"✓ Adapter LoRA sauvegardé: {adapter_save_path}")
print(f"  Fichiers:")
for file in adapter_save_path.iterdir():
    print(f"    - {file.name}")

## 8. Test d'inférence - Avant vs Après

In [ ]:
# Questions de test DataCenter
test_prompts = [
    "How do I check CPU temperature on a Linux server?",
    "What is the best backup strategy?",
    "How do I configure SSH key authentication?"
]

def generate_response(model, tokenizer, prompt, max_new_tokens=100):
    """Générer une réponse du modèle"""
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = inputs.to('cuda')
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Retirer la partie input de la réponse
    response = response.replace(prompt, "").strip()
    return response

print("="*80)
print("RÉSULTATS D'INFÉRENCE - Modèle Fine-tuné LoRA")
print("="*80)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nQuestion {i}: {prompt}")
    print("-" * 80)
    response = generate_response(model, tokenizer, prompt)
    print(f"Réponse: {response[:300]}..." if len(response) > 300 else f"Réponse: {response}")
    print()

## 9. Charger et utiliser le modèle fine-tuné

In [ ]:
# Exemple d'utilisation en production
from peft import AutoPeftModelForCausalLM

# Charger le modèle avec l'adapter LoRA
adapter_path = ADAPTER_DIR / 'qwen-datacenter-lora'

print(f"Chargement du modèle fine-tuné: {adapter_path}")

# Option 1: Charger avec PEFT (adapter séparé)
model_tuned = AutoPeftModelForCausalLM.from_pretrained(
    str(adapter_path),
    device_map="auto" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)

print(f"✓ Modèle fine-tuné chargé")

# Test
test_prompt = "How do I troubleshoot high memory usage?"
print(f"\nQuestion: {test_prompt}")
print("-" * 60)
response = generate_response(model_tuned, tokenizer, test_prompt)
print(f"Réponse: {response[:400]}..." if len(response) > 400 else f"Réponse: {response}")

## 10. Rapport et recommandations

In [ ]:
print("\n" + "="*80)
print("RAPPORT DE FINE-TUNING")
print("="*80)

print(f"""
RÉSUMÉ DE L'ENTRAÎNEMENT:
  - Modèle de base: Qwen2.5-0.5B-Instruct
  - Technique: LoRA (Low-Rank Adaptation)
  - LoRA rank (r): 16
  - Paramètres entraînables: ~2-3% du modèle total
  - Epochs: 3
  - Dataset: 19 exemples DataCenter Q&A

AVANTAGES DE LoRA:
  ✓ Économe en mémoire: ~2GB vs 24GB pour fine-tuning complet
  ✓ Rapide: Entraînement en minutes vs heures
  ✓ Adapter léger: ~5MB vs 2GB pour le modèle complet
  ✓ Modulaire: Charger/décharger plusieurs adapters

ADAPTERS SAUVEGARDÉS:
  - Chemin: {adapter_save_path}
  - Taille totale: ~5MB
  - Utilisation: PeftModel.from_pretrained()

PROCHAINES ÉTAPES EN PRODUCTION:
  1. Augmenter le dataset à 500-1000 exemples
  2. Valider sur des cas de test indépendants
  3. Mesurer les métriques: BLEU, ROUGE, etc.
  4. Intégrer dans l'API de chat
  5. Monitorer les performances

AMÉLIORATIONS FUTURES:
  - Utiliser un modèle plus grand (Mistral-7B) si GPU disponible
  - Ajouter validation set pour early stopping
  - Augmenter les données avec du data augmentation
  - Implémenter RLHF (Reinforcement Learning from Human Feedback)
""")